## 데이터 수집 자동화

### 안정적 수집과 저장

##### 수집 오류 유형
- 요소 없음 : CSS선택자가 틀렸거나 화면에 없음 > 예외처리
- 로딩 지연 : 페이지가 늦게 뜸 > 명시적 대기
- 클릭 실패 : 버튼이 아직 클릭 불가 > 명시적 대기
- 데이터 누락 : 일부 항목이 페이지에 없음 > 기본값 처리
- 중복 수집 : 같은 데이터가 포함 > 판다스 중복처리
- 저장 실패 : 파일 존재여부, 권한여부 > 파일명 변경, 예외처리

In [17]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [21]:
try:    # 실행중 오류가 날 것 같은 소스코드
    price = input('가격입력 > ')
    print(int(price))
except Exception as e: # 실행중 오류가 발생하면 처리할 부분
    print(f'가격은 숫자만 입력할 수 있습니다.')
finally: # 실행중 오류가 나든 안나든 무조건 실행하는 부분
    print('프로그램 종료')

44444
프로그램 종료


In [22]:
driver = webdriver.Chrome()

try:
    driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')  # 웹사이트 오픈
    print(driver.title)  # 웹사이트의 head 영역에 있는 탭이름(title) 가져오기
except Exception as e:
    print(f'실행 중 오류발생! : {e.args}')
finally:
    driver.quit()

게이밍PC 전체 : 다나와 가격비교


##### find_element 문제점
- find_elements() : 실행중 오류 발생 X
- find_element() : 실행중 오류 발생 가능성 높음
    - 함수를 만드는 게 유리

In [23]:
# 안전하게 요소 추출함수
def get_text_safe(parent, selector, default=''):
    try:
        return parent.find_element(By.CSS_SELECTOR, selector).text
    except:
        return default  # 내가 지정한 기본값이 그냥 리턴

In [24]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.danawa.com/')

wait.until(
    EC.visibility_of_element_located((By.ID, 'danawa_footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="c66cd4b1bc95895075398ac270414e3c", element="f.A3BCFEFE85E370D6B84530CC98951D89.d.10ECF800115EB0B4656E766E8D4F2C62.e.76")>

In [25]:
get_text_safe(driver, 'div.item > a .txt', 'None')

'최근'

##### 명시적 대기와 타임 아웃
- 명시적 대기를 하더라도 시간을 초과하면 실행 중 오류가 발생
- 실행 중 오류 잡으려면 try ~ except 사용 필요

##### DB에 저장

- 이전 저장한 다나와 데이터 사용

In [26]:
df = pd.read_csv('../data/danawa_게이밍PC.csv', encoding='utf-8')
df = df.drop(columns=["Unnamed: 0"], errors='ignore')

In [27]:
df

,index,computer_name,computer_spec,price_spec,price
0,0,영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080,9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W...,"32GB, M.2 1TB",4529770.0
1,1,영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060,5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / ...,"16GB, M.2 256GB",1209640.0
2,2,포유컴퓨터 퍼포먼스PC 47 R5 7500F RTX5060Ti,7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 700...,"16GB, M.2 500GB",1579200.0
3,3,MSI MAG Codex Z2 A9NVR A18 R7-9800X3D RTX 5070...,9800X3D / RTX 5070 Ti / (AMD) B850 / OS미포함 / 8...,"32GB, M.2 1TB",3943540.0
4,4,MSI MAG CODEX Z2 A9NVP MA09 R5-9600X RTX 5070 게이밍,9600X / RTX 5070 / (AMD) B850 / OS미포함 / 750W /...,"16GB, M.2 1TB",2545860.0
...,...,...,...,...,...
895,895,삼성전자 데스크탑 Tower DM500THZ-Z9ADZ RTX5050 Win11Pro,285 / RTX 5050 / (인텔) B860 / 윈도우11 프로 / 500W /...,"16GB, M.2 4TB",3400000.0
896,896,다나와표준PC 게임용 261015,7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 750...,NaN,NaN
897,897,이그닉 리트 빅타워 9759 RT-BTP00042,9700X / RTX 5090 / (AMD) B650 / OS미포함 / 1250W ...,"32GB, M.2 1TB",9599000.0
898,898,몬스타 몬스타PC 크리에이터 프로,270K Plus / RTX 5070 Ti / (인텔) B860 / OS미포함 / ...,"64GB, M.2 1TB",4920000.0


#### DB 저장

- SQLAIchemy 패키지 사용. SQL 쿼리문 없이 CRUD 가능
- `'pip install SQLAlchemy`

In [14]:
!pip install SQLAlchemy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 14.9 MB/s  0:00:00


In [ ]:
from datetime import datetime
from getpass import getpass

#SQLAlchemy
import psycopg
from sqlalchemy import create_engine, URL, Text, BigInteger, Integer

In [28]:
df.head()

,index,computer_name,computer_spec,price_spec,price
0,0,영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080,9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W...,"32GB, M.2 1TB",4529770.0
1,1,영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060,5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / ...,"16GB, M.2 256GB",1209640.0
2,2,포유컴퓨터 퍼포먼스PC 47 R5 7500F RTX5060Ti,7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 700...,"16GB, M.2 500GB",1579200.0
3,3,MSI MAG Codex Z2 A9NVR A18 R7-9800X3D RTX 5070...,9800X3D / RTX 5070 Ti / (AMD) B850 / OS미포함 / 8...,"32GB, M.2 1TB",3943540.0
4,4,MSI MAG CODEX Z2 A9NVP MA09 R5-9600X RTX 5070 게이밍,9600X / RTX 5070 / (AMD) B850 / OS미포함 / 750W /...,"16GB, M.2 1TB",2545860.0


In [29]:
# 저장할 테이블명
table_name = 'danawa_gaming_pc_' + datetime.now().strftime('%y%m%d')
print(table_name)


danawa_gaming_pc_261006


In [ ]:
# PostgreSQL 연결정보
db_url = URL.create(
    'postgresql+psycopg',
    username='postgres', 
    password='123456',
    port=5432,
    database='postgres'
)

In [37]:
engine = create_engine(db_url)

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   index          900 non-null    int64  
 1   computer_name  900 non-null    str    
 2   computer_spec  900 non-null    str    
 3   price_spec     897 non-null    str    
 4   price          897 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 35.3 KB


In [38]:
# 테이블 생성하고 DataFrame 데이터 저장
df.to_sql(
    name=table_name,
    con=engine,
    if_exists='fail',
    index=False,
    dtype={
        'index': BigInteger(),
        'computer_name': Text(),
        'computer_spec': Text(),
        'price_spec': Text(),
        'price': Integer()
    }
)

engine.dispose() # 사용 후 해제

print(f'{len(df)}건을 저장했습니다. ')

900건을 저장했습니다. 


#### 스케줄링

- 정해진 시간마다 반복 수행하도록 하는 것
- OS 마다 방식이 다름
    - Windows : 작업 스케줄러
    - Linux : Cron
    - Python : schedule 라이브러리 가능

#### 윈도우 작업 스케줄러

- 윈도우키 > 작업 스케줄러 검색
- `기본작업 만들기`...클릭
- 기본작업 만들기 > 이름과 설명(옵션) 추가
- 트리거 > 작업을 매일로 선택 
    - 작업 내용은 상황에 맞게 변경
- 작업 > 프로그램 시작
- 동작 탭 > 새로 만들기 > 
    - 동작 : 프로그램 시작
    - 프로그램 스크립트 : .bat 파일 위치선택
    - 시작위치 : D:\SourceBank\ai-datapipeline-2026\chapt04 경로 입력
    - 마침 창 체크표시
- 속성 창 오픈 
    - 보안 옵션 : Administrators(관리자 그룹)을 선택하고 사용자 변경
    - 가장 높은 수준의 권한으로 실행 체크